In [ ]:
def eval_better(solve, verbose=0):
    items_old = torch.full((BOX_SIZE, BOX_SIZE, len(Tile)), NEG_INF)
    items_old[INPUT_POS][:] = 0.0

    for iter in range(round(BOX_SIZE*1.8)):
        log_total_mass = items_old.max(dim=-1)[0]
        log_total_mass = torch.clamp(log_total_mass, min=NEG_INF)

        items_ins = items_old.clone()
        for i in range(len(Tile)):
            tile_type = Tile(i)
            t_data = TILE_DATA[tile_type]
            idx = t_data.can_give_to

            if not t_data.can_pull():
                continue
            diff_y = t_data._output[1]-t_data._pull_pos[1]
            diff_x = t_data._output[0]-t_data._pull_pos[0]

            inp_y = slice(max(diff_y,0), BOX_SIZE-max(-diff_y,0))
            inp_x = slice(max(-diff_x,0), BOX_SIZE-max(diff_x,0))

            out_y = slice(max(-diff_y,0), BOX_SIZE-max(diff_y,0))
            out_x = slice(max(diff_x,0), BOX_SIZE-max(-diff_x,0))

            solv_y = slice(max(diff_y,0)+t_data._pull_pos[1], BOX_SIZE-max(-diff_y,0)+t_data._pull_pos[1])
            solv_x = slice(max(-diff_x,0)-t_data._pull_pos[0], BOX_SIZE-max(diff_x,0)-t_data._pull_pos[0])

            # print(tile_type, inp_y,inp_x,out_y,out_x,solv_y,solv_x)
            
            src = (items_old[inp_y, inp_x, i] + solve[solv_y,solv_x,i]).unsqueeze(-1).repeat(1, 1, len(idx))
            items_ins[out_y,out_x, idx] = torch.logaddexp(items_ins[out_y,out_x, idx], src)
            items_ins[inp_y, inp_x, idx] = logsubexp(items_ins[inp_y, inp_x, idx], src)
        items_old = items_ins.clone()

        mul_self = items_old + solve # + log_self_fraction.unsqueeze(-1)

        items = torch.full((BOX_SIZE, BOX_SIZE, len(Tile)), NEG_INF)
        for i in range(len(Tile)):
            tile_type = Tile(i)
            t_data: TileData = TILE_DATA[tile_type]

            if len(t_data._output) != 0:
                idx = t_data.can_give_to
                if not t_data.can_pull():
  

                    inp_y = slice(max(t_data._output[1],0), BOX_SIZE-max(-t_data._output[1],0))
                    inp_x = slice(max(-t_data._output[0],0), BOX_SIZE-max(t_data._output[0],0))

                    out_y = slice(max(-t_data._output[1],0), BOX_SIZE-max(t_data._output[1],0))
                    out_x = slice(max(t_data._output[0],0), BOX_SIZE-max(-t_data._output[0],0))

                    src = mul_self[inp_y,inp_x, i].unsqueeze(-1).repeat(1, 1, len(idx))
                    items[out_y, out_x, idx] = torch.logaddexp(items[out_y, out_x, idx], src)

        if verbose>5:
            plt.matshow(items.detach().max(-1)[0].exp().numpy(),vmin=0,vmax=1)
            plt.show()
        
        sum_items = items.max(-1)[0].exp().sum().item()
        assert sum_items <= 1.15, f'Sum should be <= 1. ({sum_items})'
        
        items_old = items


    return items


# eval(solve,verbose=10)